# Tech Layoffs Data Cleaning in SQL

I have copied this notework **[Tech Layoffs Data Cleaning in SQL] (https://www.kaggle.com/code/wilfridawere/tech-layoffs-data-cleaning-in-sql)** and take it as a starting point for a SQL exercise. The original notebook used *SQL (sqlite3)* to clean a dataset I put on Kaggle **[Tech layoffs 2020 - 2024](https://www.kaggle.com/datasets/ulrikeherold/tech-layoffs-2020-2024)** but a updated version. 

In [1]:
import numpy as np
import pandas as pd
import sqlite3 # for using SQL commands

In [3]:
# Load the data
tech_layoffs = pd.read_csv('/Users/ulrike_imac_air/projects/DataScienceProjects/tech_layoffs/tech_layoffs_csv/tech_layoffs_til_2026.csv')
df = tech_layoffs.copy() # make a copy
df.head()

,Nr,Company,Location_HQ,Region,USState,Country,Continent,Laid_Off,Date_layoffs,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs,Industry,Stage,Money_Raised_in__mil,Year,latitude,longitude
0,1,Tamara Mellon,Los Angeles,other,California,USA,North America,20.0,2020-03-12,40.0,50.0,30.0,Retail,Series C,90.0,2020,34.053691,-118.242766
1,2,HopSkipDrive,Los Angeles,other,California,USA,North America,8.0,2020-03-13,10.0,80.0,72.0,Transportation,Unknown,45.0,2020,34.053691,-118.242766
2,3,Panda Squad,San Francisco,San Francisco Bay Area,California,USA,North America,6.0,2020-03-13,75.0,8.0,2.0,Consumer,Seed,1.0,2020,37.779259,-122.419329
3,4,Help.com,Austin,other,Texas,USA,North America,16.0,2020-03-16,100.0,16.0,0.0,Support,Seed,6.0,2020,30.271129,-97.743700
4,5,Inspirato,Denver,other,Colorado,USA,North America,130.0,2020-03-16,22.0,591.0,461.0,Travel,Series C,79.0,2020,39.739236,-104.984862


In [4]:
df.info()  # 1839 rows and 18 columns

<class 'pandas.DataFrame'>
RangeIndex: 2592 entries, 0 to 2591
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Nr                           2592 non-null   int64  
 1   Company                      2592 non-null   str    
 2   Location_HQ                  2590 non-null   str    
 3   Region                       2592 non-null   str    
 4   USState                      2591 non-null   str    
 5   Country                      2592 non-null   str    
 6   Continent                    2592 non-null   str    
 7   Laid_Off                     2154 non-null   float64
 8   Date_layoffs                 2592 non-null   str    
 9   Percentage                   2032 non-null   float64
 10  Company_Size_before_Layoffs  1805 non-null   float64
 11  Company_Size_after_layoffs   1893 non-null   float64
 12  Industry                     2592 non-null   str    
 13  Stage                        

The columns `Laid_Off`, `Percentage`, `Company_Size_before_Layoffs`, `Company_Size_after_layoffs` have missing values so I will inspect them. This helps decide whether to impute or drop those rows with missing values. I don't need the `Money_Raised_in_$_mil` column

# Connect to a Database and convert the Dataframe to a Table

In [5]:
# Connect to the database
conn = sqlite3.connect('tech_layoffs_data.db')

In [6]:
# Create 'tech_layoffs' table
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='tech_layoffs'")
table_exists = len(cursor.fetchall()) > 0

if not table_exists:
  # Load data to 'tech_layoffs' table only if it doesn't exist
  df.to_sql('tech_layoffs', conn, index=False)
  print("Table 'tech_layoffs' created and data loaded successfully.")
else:
  print("Table 'tech_layoffs' already exists. Skipping data loading.")

Table 'tech_layoffs' already exists. Skipping data loading.


In [7]:
# Now I will use SQL commands to Clean data from the 'tech_layoffs' table
# pd.read_sql_query is an optimized approach for working with pandas and sqlite3

# Inspect the first_10_rows
first_10_rows = pd.read_sql_query("""SELECT *
                                   FROM tech_layoffs
                                   LIMIT 10;""", conn)
first_10_rows

,Nr,Company,Location_HQ,Region,USState,Country,Continent,Laid_Off,Date_layoffs,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs,Industry,Stage,Money_Raised_in__mil,Year,latitude,longitude
0,1,Tamara Mellon,Los Angeles,other,California,USA,North America,20.0,2020-03-12,40.0,50.0,30.0,Retail,Series C,90.0,2020,34.053691,-118.242766
1,2,HopSkipDrive,Los Angeles,other,California,USA,North America,8.0,2020-03-13,10.0,80.0,72.0,Transportation,Unknown,45.0,2020,34.053691,-118.242766
2,3,Panda Squad,San Francisco,San Francisco Bay Area,California,USA,North America,6.0,2020-03-13,75.0,8.0,2.0,Consumer,Seed,1.0,2020,37.779259,-122.419329
3,4,Help.com,Austin,other,Texas,USA,North America,16.0,2020-03-16,100.0,16.0,0.0,Support,Seed,6.0,2020,30.271129,-97.743700
4,5,Inspirato,Denver,other,Colorado,USA,North America,130.0,2020-03-16,22.0,591.0,461.0,Travel,Series C,79.0,2020,39.739236,-104.984862
5,6,Flytedesk,Boulder,other,Colorado,USA,North America,4.0,2020-03-18,20.0,20.0,16.0,Marketing,Seed,4.0,2020,40.014986,-105.270545
6,7,Remote Year,Chicago,other,Illinois,USA,North America,50.0,2020-03-19,50.0,100.0,50.0,Travel,Series B,17.0,2020,41.875562,-87.624421
7,8,CTO.ai,Vancouver,Cascadia,British Columbia,Canada,North America,30.0,2020-03-20,50.0,60.0,30.0,Infrastructure,Seed,7.0,2020,49.260872,-123.113952
8,9,Flywheel Sports,New York City,other,New York,USA,North America,784.0,2020-03-20,98.0,800.0,16.0,Fitness,Acquired,120.0,2020,40.712728,-74.006015
9,10,Compass,New York City,other,New York,USA,North America,375.0,2020-03-23,15.0,2500.0,2125.0,Real Estate,Series G,1600.0,2020,40.712728,-74.006015


# Data Cleaning

# Identify Spelling Errors

In [8]:
# Inspect the Industry column 
industry = pd.read_sql_query("""
SELECT DISTINCT Industry
FROM tech_layoffs;
""",conn)

industry

,Industry
0,Retail
1,Transportation
2,Consumer
3,Support
4,Travel
...,...
113,Advertising Services
114,Meat Technologies
115,E-Learning
116,Conversational chatbot tools


# Duplicate Rows

In [9]:
# Identify rows that have duplicate values in all these columns. I will drop the duplicate rows later
# Company, Location_HQ, Country, Laid_Off, Date_layoffs, Industry

duplicate_rows = pd.read_sql_query("""
SELECT *
FROM (
  SELECT Company, Location_HQ, Country, Laid_Off, Date_layoffs, Industry,
         ROW_NUMBER() OVER (PARTITION BY Company, Location_HQ, Country, Laid_Off, Date_layoffs, Industry) AS row_num
  FROM tech_layoffs
) AS duplicates  
WHERE row_num > 1;
""", conn)

duplicate_rows

,Company,Location_HQ,Country,Laid_Off,Date_layoffs,Industry,row_num
0,BeReal,Paris,France,NaN,2024-06-25,Consumer,2
1,Bluevine,Tel Aviv,Israel,NaN,2024-06-20,Finance,2
2,C2FO,Kansas City,USA,16.0,2024-06-18,Finance,2
3,Chegg,Santa Clara,USA,441.0,2024-06-17,Education,2
4,Criteo,Paris,France,140.0,2024-04-12,Marketing,2
5,Etsy,New York City,USA,225.0,2023-12-13,Retail,2
6,Flutterwave,San Francisco,USA,30.0,2024-06-24,Finance,2
7,InSightec,Haifa,Israel,100.0,2023-12-19,Healthcare,2
8,Moxion Power,Richmond,USA,101.0,2024-06-26,Energy,2
9,Planet,San Francisco,USA,180.0,2024-06-26,Aerospace,2


In [10]:
# Inspect any of the duplicate rows
Criteo = pd.read_sql_query("""
SELECT Company, Location_HQ, Country, Laid_Off,Date_layoffs, Industry
FROM tech_layoffs
WHERE Company LIKE 'Criteo';
""",conn)

Criteo

,Company,Location_HQ,Country,Laid_Off,Date_layoffs,Industry
0,Criteo,Paris,France,140.0,2024-04-12,Marketing
1,Criteo,Paris,France,140.0,2024-04-12,Marketing


In [11]:
# Inspect another duplicate row
Etsy = pd.read_sql_query("""
SELECT Company, Location_HQ, Country, Laid_Off,Date_layoffs, Industry
FROM tech_layoffs
WHERE Company LIKE 'Etsy'
""",conn)

Etsy

,Company,Location_HQ,Country,Laid_Off,Date_layoffs,Industry
0,Etsy,New York City,USA,225.0,2023-12-13,Retail
1,Etsy,New York City,USA,225.0,2023-12-13,Retail


In [12]:
# Inspect another duplicate row
Tome = pd.read_sql_query("""
SELECT Company, Location_HQ, Country, Laid_Off,Date_layoffs, Industry
FROM tech_layoffs
WHERE Company LIKE 'Tome'
""",conn)

Tome

,Company,Location_HQ,Country,Laid_Off,Date_layoffs,Industry
0,Tome,San Francisco,USA,12.0,2024-04-16,AI
1,Tome,San Francisco,USA,12.0,2024-04-16,AI
2,Tome,San Francisco,USA,12.0,2024-10-02,Sales


In [13]:
# Inspect another duplicate row
Meta = pd.read_sql_query("""
SELECT Company, Location_HQ, Country, Laid_Off,Date_layoffs, Industry
FROM tech_layoffs
WHERE Company LIKE 'Meta'
""",conn)

Meta

,Company,Location_HQ,Country,Laid_Off,Date_layoffs,Industry
0,Meta,Menlo Park,USA,11000.0,2022-11-09,Consumer
1,Meta,Menlo Park,USA,10000.0,2023-03-14,Consumer
2,Meta,Menlo Park,USA,NaN,2024-03-06,Consumer
3,Meta,Menlo Park,USA,50.0,2024-06-12,Software Development
4,Meta,Menlo Park,USA,NaN,2024-10-16,Consumer
5,Meta,Menlo Park,USA,3600.0,2025-02-10,Consumer
6,Meta,Menlo Park,USA,100.0,2025-04-24,Consumer
7,Meta,Menlo Park,USA,600.0,2025-10-22,Software Development


# Missing Values

In [14]:
# Missing values in Laid_Off, Percentage, Company_Size_before_Layoffs, Company_Size_after_layoffs columns
missing_values = pd.read_sql_query("""
SELECT Laid_Off, Percentage, Company_Size_before_Layoffs, Company_Size_after_layoffs
FROM tech_layoffs
WHERE Laid_Off IS NULL
OR Percentage IS NULL 
OR Company_Size_before_Layoffs IS NULL 
OR Company_Size_after_layoffs IS NULL;
""",conn)

for col in missing_values.columns:
    print(f"Missing values in '{col}': {missing_values[col].isnull().sum()}")

print() # blank line
missing_values.head(10) # the first 10 rows

Missing values in 'Laid_Off': 372
Missing values in 'Percentage': 449
Missing values in 'Company_Size_before_Layoffs': 643
Missing values in 'Company_Size_after_layoffs': 555



,Laid_Off,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs
0,NaN,NaN,NaN,NaN
1,NaN,100.0,NaN,0.0
2,NaN,100.0,NaN,0.0
3,45.0,NaN,NaN,NaN
4,111.0,NaN,NaN,NaN
5,31.0,NaN,NaN,NaN
6,109.0,NaN,NaN,NaN
7,NaN,15.0,NaN,NaN
8,NaN,20.0,NaN,NaN
9,NaN,100.0,NaN,0.0


# The Problem with Imputation

In this case, the columns `Laid_Off`, `Percentage`, `Company_Size_before_Layoffs`, and `Company_Size_after_layoffs` are all **related**.

Imagine these rows represent companies that went through layoffs.  Since we don't have all the information *(number laid off, company size before/after)*,  trying to guess those missing values would be like making up a story.  It's better to remove these rows and focus on the companies where we have complete data for a more accurate analysis.

In [15]:
# FILTER the data, EXCLUDING rows with missing values in the specified columns
Cleaned_tech_layoffs = pd.read_sql_query("""
SELECT *
FROM tech_layoffs
WHERE Laid_Off IS NOT NULL
AND Percentage IS NOT NULL 
AND Company_Size_before_Layoffs IS NOT NULL 
AND Company_Size_after_layoffs IS NOT NULL;
""",conn)

print(Cleaned_tech_layoffs.shape) # 1572 clean rows remain after Filtering 
print() # blank line
Cleaned_tech_layoffs.head()

(1754, 18)



,Nr,Company,Location_HQ,Region,USState,Country,Continent,Laid_Off,Date_layoffs,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs,Industry,Stage,Money_Raised_in__mil,Year,latitude,longitude
0,1,Tamara Mellon,Los Angeles,other,California,USA,North America,20.0,2020-03-12,40.0,50.0,30.0,Retail,Series C,90.0,2020,34.053691,-118.242766
1,2,HopSkipDrive,Los Angeles,other,California,USA,North America,8.0,2020-03-13,10.0,80.0,72.0,Transportation,Unknown,45.0,2020,34.053691,-118.242766
2,3,Panda Squad,San Francisco,San Francisco Bay Area,California,USA,North America,6.0,2020-03-13,75.0,8.0,2.0,Consumer,Seed,1.0,2020,37.779259,-122.419329
3,4,Help.com,Austin,other,Texas,USA,North America,16.0,2020-03-16,100.0,16.0,0.0,Support,Seed,6.0,2020,30.271129,-97.743700
4,5,Inspirato,Denver,other,Colorado,USA,North America,130.0,2020-03-16,22.0,591.0,461.0,Travel,Series C,79.0,2020,39.739236,-104.984862


In [15]:
# Drop DUPLICATE rows from the Dataframe, based on the columns used earlier when inspecting Duplicates
Cleaned_tech_layoffs = Cleaned_tech_layoffs.drop_duplicates(subset=['Company', 'Location_HQ', 'Country', 'Laid_Off',
                                                                    'Date_layoffs','Industry'], keep='first') # keep first instances

print(Cleaned_tech_layoffs.shape) # 1569 rows remain after dropping duplicate rows

(1745, 18)


In [16]:
# Save the cleaned DataFrame to a CSV file
Cleaned_tech_layoffs.to_csv('Cleaned_tech_layoffs.csv', index=False)
print('Cleaned_tech_layoffs.csv saved.')

Cleaned_tech_layoffs.csv saved.


In [16]:
# Close the connection
conn.commit()  # Save changes if any
conn.close()